# SecurePay Vision - Model Training & Evaluation

**Final Project: AI-Powered Fraud Detection for UMKM**

Notebook ini berisi proses training model machine learning untuk deteksi fraud transaksi digital.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_curve, auc,
    precision_score, recall_score, f1_score, accuracy_score
)
import joblib
import warnings
warnings.filterwarnings('ignore')

# Style settings
plt.style.use('dark_background')
sns.set_palette('husl')

print('Libraries loaded successfully!')

## 1. Load Dataset

In [ ]:
# Load dataset
df = pd.read_csv('../dataset/transactions.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nClass distribution:')
print(df['is_fraud'].value_counts())
print(f'\nFraud rate: {df["is_fraud"].mean():.1%}')
df.head()

## 2. Feature Engineering

In [ ]:
# Feature selection for ML models
FEATURES = ['amount_log', 'hour', 'day_of_week', 'amount_zscore', 
            'is_odd_hours', 'is_weekend', 'amount_percentile']

# Encode payment method
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['payment_encoded'] = le.fit_transform(df['payment_method'])
FEATURES.append('payment_encoded')

X = df[FEATURES].values
y = df['is_fraud'].values

print(f'Features: {FEATURES}')
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

## 3. Data Preprocessing

In [ ]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split: use normal data only for training (unsupervised)
X_normal = X_scaled[y == 0]
X_fraud = X_scaled[y == 1]

print(f'Normal samples: {len(X_normal)}')
print(f'Fraud samples: {len(X_fraud)}')
print(f'Contamination ratio: {len(X_fraud)/len(X_scaled):.3f}')

## 4. Train Isolation Forest

In [ ]:
contamination = len(X_fraud) / len(X_scaled)

# Isolation Forest
iso_forest = IsolationForest(
    n_estimators=200,
    contamination=contamination,
    random_state=42,
    n_jobs=-1
)
iso_forest.fit(X_normal)  # Train on normal data only

# Predict on full dataset
iso_pred = iso_forest.predict(X_scaled)
iso_pred_binary = (iso_pred == -1).astype(int)  # -1 = anomaly = fraud

print('Isolation Forest Results:')
print(classification_report(y, iso_pred_binary, target_names=['Normal', 'Fraud']))

## 5. Train Local Outlier Factor

In [ ]:
# Local Outlier Factor
lof = LocalOutlierFactor(
    n_neighbors=20,
    contamination=contamination,
    novelty=True,
    n_jobs=-1
)
lof.fit(X_normal)

lof_pred = lof.predict(X_scaled)
lof_pred_binary = (lof_pred == -1).astype(int)

print('Local Outlier Factor Results:')
print(classification_report(y, lof_pred_binary, target_names=['Normal', 'Fraud']))

## 6. Train One-Class SVM

In [ ]:
# One-Class SVM
svm = OneClassSVM(
    nu=contamination,
    kernel='rbf',
    gamma='scale'
)
svm.fit(X_normal)

svm_pred = svm.predict(X_scaled)
svm_pred_binary = (svm_pred == -1).astype(int)

print('One-Class SVM Results:')
print(classification_report(y, svm_pred_binary, target_names=['Normal', 'Fraud']))

## 7. Ensemble Voting

In [ ]:
# Weighted ensemble: IF (50%) + LOF (30%) + SVM (20%)
ensemble_score = (
    0.5 * iso_pred_binary +
    0.3 * lof_pred_binary +
    0.2 * svm_pred_binary
)
ensemble_pred = (ensemble_score >= 0.5).astype(int)

print('Ensemble Model Results:')
print(classification_report(y, ensemble_pred, target_names=['Normal', 'Fraud']))

print(f'\nMetrics Summary:')
print(f'Accuracy:  {accuracy_score(y, ensemble_pred):.4f}')
print(f'Precision: {precision_score(y, ensemble_pred):.4f}')
print(f'Recall:    {recall_score(y, ensemble_pred):.4f}')
print(f'F1-Score:  {f1_score(y, ensemble_pred):.4f}')

## 8. Confusion Matrix Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Confusion Matrices - SecurePay Vision', fontsize=16, color='white')

models_results = [
    ('Isolation Forest', iso_pred_binary),
    ('Local Outlier Factor', lof_pred_binary),
    ('One-Class SVM', svm_pred_binary)
]

for ax, (name, preds) in zip(axes, models_results):
    cm = confusion_matrix(y, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal', 'Fraud'],
                yticklabels=['Normal', 'Fraud'], ax=ax)
    ax.set_title(name, color='white')
    ax.set_ylabel('True Label', color='white')
    ax.set_xlabel('Predicted Label', color='white')

plt.tight_layout()
plt.savefig('../reports/confusion_matrices.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('Confusion matrices saved!')

## 9. ROC Curve

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(8, 6))

colors = ['#00d4ff', '#00ffcc', '#7c3aed']
for (name, preds), color in zip(models_results, colors):
    fpr, tpr, _ = roc_curve(y, preds)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2,
            label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'gray', linestyle='--', lw=1, label='Random Classifier')
ax.set_xlim([0.0, 1.0])
ax.set_ylim([0.0, 1.05])
ax.set_xlabel('False Positive Rate', color='white')
ax.set_ylabel('True Positive Rate', color='white')
ax.set_title('ROC Curves - SecurePay Vision AI Models', color='white', fontsize=14)
ax.legend(loc='lower right', facecolor='#1a1f35', edgecolor='#00d4ff')
ax.grid(alpha=0.2)

plt.tight_layout()
plt.savefig('../reports/roc_curves.png', dpi=150, bbox_inches='tight',
            facecolor='#0d1117')
plt.show()
print('ROC curves saved!')

## 10. Save Models

In [ ]:
import os
os.makedirs('../trained_models', exist_ok=True)

# Save models
joblib.dump(iso_forest, '../trained_models/isolation_forest.pkl')
joblib.dump(lof, '../trained_models/lof_model.pkl')
joblib.dump(svm, '../trained_models/svm_model.pkl')
joblib.dump(scaler, '../trained_models/scaler.pkl')
joblib.dump(le, '../trained_models/label_encoder.pkl')

# Save feature names
import json
with open('../trained_models/features.json', 'w') as f:
    json.dump(FEATURES, f)

print('All models saved successfully!')
print('Models saved to: ../trained_models/')